# 🏒 NHL Game Data — Phase 2: Gold Layer (Star Schema) & Business KPIs
**Microsoft Fabric | Junior Data Engineer Final Project**

This notebook builds the **Gold Layer** — a Star Schema optimised for Power BI.

```
        dim_team
           │
dim_player─┤
           │
     fact_game_performance ── dim_date
           │
     fact_player_stats
           │
     fact_play_events
```

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.conf.set('spark.sql.shuffle.partitions', '8')

# Load silver tables
game           = spark.table('silver_game')
teams_stats    = spark.table('silver_game_teams_stats')
skater_stats   = spark.table('silver_game_skater_stats')
goalie_stats   = spark.table('silver_game_goalie_stats')
plays          = spark.table('silver_game_plays')
player_info    = spark.table('silver_player_info')
team_info      = spark.table('silver_team_info')

print('Silver tables loaded ✅')

---
## 1. Dimension Tables

In [ ]:
# ── dim_team ──────────────────────────────────────────────────────────────────
dim_team = (
    team_info
    .select(
        F.col('team_id'),
        F.col('franchiseId').alias('franchise_id'),
        F.col('shortName').alias('city'),
        F.col('teamName').alias('team_name'),
        F.col('abbreviation'),
        F.col('link').alias('nhl_api_link')
    )
    .dropDuplicates(['team_id'])
)
dim_team.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable('dim_team')
print(f'dim_team: {dim_team.count()} rows ✅')

In [ ]:
# ── dim_player ────────────────────────────────────────────────────────────────
dim_player = (
    player_info
    .select(
        F.col('player_id'),
        F.col('firstName').alias('first_name'),
        F.col('lastName').alias('last_name'),
        F.concat(F.col('firstName'), F.lit(' '), F.col('lastName')).alias('full_name'),
        F.col('nationality'),
        F.col('birthCity').alias('birth_city'),
        F.col('primaryPosition').alias('position'),
        F.col('birthDate').alias('birth_date'),
        F.col('height_cm').alias('height_cm'),
        F.col('weight').alias('weight_lbs'),
        F.col('shootsCatches').alias('shoots_catches')
    )
    .dropDuplicates(['player_id'])
)
dim_player.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable('dim_player')
print(f'dim_player: {dim_player.count()} rows ✅')

In [ ]:
# ── dim_date ──────────────────────────────────────────────────────────────────
dim_date = (
    game
    .select(F.col('date_time'))
    .dropna()
    .withColumn('date_key', F.date_format('date_time', 'yyyyMMdd').cast(IntegerType()))
    .withColumn('date', F.to_date('date_time'))
    .withColumn('year', F.year('date_time'))
    .withColumn('month', F.month('date_time'))
    .withColumn('month_name', F.date_format('date_time', 'MMMM'))
    .withColumn('day', F.dayofmonth('date_time'))
    .withColumn('day_of_week', F.dayofweek('date_time'))
    .withColumn('day_name', F.date_format('date_time', 'EEEE'))
    .withColumn('quarter', F.quarter('date_time'))
    .withColumn('week_of_year', F.weekofyear('date_time'))
    .withColumn('is_weekend',
        F.when(F.dayofweek('date_time').isin([1, 7]), True).otherwise(False))
    .select('date_key','date','year','month','month_name','day',
            'day_of_week','day_name','quarter','week_of_year','is_weekend')
    .dropDuplicates(['date_key'])
    .orderBy('date_key')
)
dim_date.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable('dim_date')
print(f'dim_date: {dim_date.count()} rows ✅')

---
## 2. Fact Tables

In [ ]:
# ── fact_game_performance ─────────────────────────────────────────────────────
# One row per game per team
fact_game_performance = (
    teams_stats
    .join(game.select(
        'game_id', 'season', 'type', 'date_time',
        'home_team_id', 'away_team_id',
        'home_goals', 'away_goals',
        'outcome', 'total_goals', 'game_result'), 'game_id', 'left')
    .withColumn('date_key', F.date_format('date_time', 'yyyyMMdd').cast(IntegerType()))
    .withColumn('is_home', F.col('home_or_away') == 'home')
    .withColumn('opponent_team_id',
        F.when(F.col('home_or_away') == 'home', F.col('away_team_id'))
         .otherwise(F.col('home_team_id')))
    .select(
        'game_id', 'team_id', 'opponent_team_id', 'season', 'type',
        'date_key', 'home_or_away', 'is_home',
        'goals', 'shots', 'hits', 'pim',
        'powerPlayOpportunities', 'powerPlayGoals',
        'faceOffWins', 'faceoffTaken',
        'takeaways', 'giveaways', 'blocked',
        'won', 'settled_in',
        'shot_pct', 'faceoff_win_pct',
        'head_coach'
    )
)
fact_game_performance.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('fact_game_performance')
print(f'fact_game_performance: {fact_game_performance.count():,} rows ✅')

In [ ]:
# ── fact_player_stats ─────────────────────────────────────────────────────────
# One row per player per game (skaters + goalies unioned)

skater_fact = (
    skater_clean
    .join(game.select('game_id','season','type','date_time'), 'game_id', 'left')
    .withColumn('date_key', F.date_format('date_time', 'yyyyMMdd').cast(IntegerType()))
    .withColumn('player_type', F.lit('skater'))
    .select(
        'game_id', 'player_id', 'team_id', 'season', 'type',
        'date_key', 'player_type',
        F.col('goals'), F.col('assists'), F.col('points'),
        F.col('shots'), F.col('hits'), F.col('blocked'),
        F.col('penaltyMinutes').alias('penalty_minutes'),
        F.col('plusMinus').alias('plus_minus'),
        F.col('giveaways'), F.col('takeaways'),
        F.col('toi_minutes'), F.col('shooting_pct'),
        F.lit(None).cast(DoubleType()).alias('save_pct'),
        F.lit(None).cast(DoubleType()).alias('goals_against_avg'),
        F.col('evenTimeOnIce').alias('even_toi'),
        F.col('powerPlayTimeOnIce').alias('pp_toi'),
        F.col('shortHandedTimeOnIce').alias('sh_toi')
    )
)

goalie_fact = (
    goalie_clean
    .join(game.select('game_id','season','type','date_time'), 'game_id', 'left')
    .withColumn('date_key', F.date_format('date_time', 'yyyyMMdd').cast(IntegerType()))
    .withColumn('player_type', F.lit('goalie'))
    .select(
        'game_id', 'player_id', 'team_id', 'season', 'type',
        'date_key', 'player_type',
        F.lit(0).cast(IntegerType()).alias('goals'),
        F.lit(0).cast(IntegerType()).alias('assists'),
        F.lit(0).cast(IntegerType()).alias('points'),
        F.col('shots').alias('shots'),
        F.lit(0).cast(IntegerType()).alias('hits'),
        F.lit(0).cast(IntegerType()).alias('blocked'),
        F.col('pim').alias('penalty_minutes'),
        F.lit(None).cast(IntegerType()).alias('plus_minus'),
        F.lit(0).cast(IntegerType()).alias('giveaways'),
        F.lit(0).cast(IntegerType()).alias('takeaways'),
        F.round(F.col('timeOnIce') / 60, 2).alias('toi_minutes'),
        F.lit(None).cast(DoubleType()).alias('shooting_pct'),
        F.col('save_pct'),
        F.col('goals_against_avg'),
        F.lit(None).cast(IntegerType()).alias('even_toi'),
        F.lit(None).cast(IntegerType()).alias('pp_toi'),
        F.lit(None).cast(IntegerType()).alias('sh_toi')
    )
)

fact_player_stats = skater_fact.union(goalie_fact)
fact_player_stats.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('fact_player_stats')
print(f'fact_player_stats: {fact_player_stats.count():,} rows ✅')

In [ ]:
# ── fact_play_events ──────────────────────────────────────────────────────────
# Filtered to key events only for performance
KEY_EVENTS = ['Goal', 'Shot', 'Penalty', 'Hit', 'Missed Shot', 'Blocked Shot',
              'Takeaway', 'Giveaway', 'Faceoff']

fact_play_events = (
    plays_clean
    .filter(F.col('event').isin(KEY_EVENTS))
    .join(game.select('game_id','season','type','date_time'), 'game_id', 'left')
    .withColumn('date_key', F.date_format('date_time', 'yyyyMMdd').cast(IntegerType()))
    .select(
        'play_id', 'game_id', 'season', 'type', 'date_key',
        F.col('team_id_for').cast(IntegerType()).alias('team_id_for'), F.col('team_id_against').cast(IntegerType()).alias('team_id_against'),
        'event', F.col('secondaryType').alias('secondary_type'),
        'period', 'period_type', 'periodTime',
        'x', 'y', 'description'
    )
)
fact_play_events.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('fact_play_events')
print(f'fact_play_events: {fact_play_events.count():,} rows ✅')

---
## 3. Gold KPI Views (for Power BI)

In [ ]:
# ── KPI 1: Season Team Standings ──────────────────────────────────────────────
standings = spark.sql("""
    SELECT
        f.season,
        t.abbreviation,
        t.team_name,
        t.city,
        COUNT(DISTINCT f.game_id)          AS games_played,
        SUM(f.won)                         AS wins,
        COUNT(DISTINCT f.game_id) - SUM(f.won) AS losses,
        ROUND(SUM(f.won) / COUNT(DISTINCT f.game_id) * 100, 1) AS win_pct,
        SUM(f.goals)                       AS goals_for,
        SUM(f.shots)                       AS total_shots,
        ROUND(AVG(f.shot_pct), 2)          AS avg_shot_pct,
        ROUND(AVG(f.faceoff_win_pct), 2)   AS avg_faceoff_win_pct,
        SUM(f.powerPlayGoals)              AS pp_goals,
        SUM(f.hits)                        AS total_hits
    FROM fact_game_performance f
    JOIN dim_team t ON f.team_id = t.team_id
    WHERE f.type = 'R'
    GROUP BY f.season, t.abbreviation, t.team_name, t.city
    ORDER BY f.season, win_pct DESC
""")
standings.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('gold_team_standings')
print(f'gold_team_standings: {standings.count()} rows ✅')

In [ ]:
# ── KPI 2: Player Season Summary ─────────────────────────────────────────────
player_season = spark.sql("""
    SELECT
        ps.season,
        ps.player_id,
        p.full_name,
        p.position,
        p.nationality,
        t.abbreviation   AS team,
        ps.player_type,
        COUNT(DISTINCT ps.game_id) AS games_played,
        SUM(ps.goals)              AS goals,
        SUM(ps.assists)            AS assists,
        SUM(ps.goals) + SUM(ps.assists) AS points,
        SUM(ps.shots)              AS shots,
        SUM(ps.hits)               AS hits,
        SUM(ps.blocked)            AS blocked_shots,
        SUM(ps.penalty_minutes)    AS penalty_minutes,
        SUM(ps.plus_minus)         AS plus_minus,
        ROUND(AVG(ps.toi_minutes), 2)  AS avg_toi_minutes,
        ROUND(AVG(ps.shooting_pct), 2) AS avg_shooting_pct,
        ROUND(AVG(ps.save_pct), 3)     AS avg_save_pct,
        ROUND(AVG(ps.goals_against_avg), 2) AS avg_gaa
    FROM fact_player_stats ps
    JOIN dim_player p ON ps.player_id = p.player_id
    JOIN dim_team t ON ps.team_id = t.team_id
    GROUP BY ps.season, ps.player_id, p.full_name, p.position,
             p.nationality, t.abbreviation, ps.player_type
    ORDER BY ps.season, points DESC
""")
player_season.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('gold_player_season_stats')
print(f'gold_player_season_stats: {player_season.count():,} rows ✅')

In [ ]:
# ── KPI 3: Home vs Away Performance Summary ───────────────────────────────────
home_away = spark.sql("""
    SELECT
        season,
        home_or_away,
        COUNT(game_id)               AS total_games,
        SUM(won)                     AS wins,
        ROUND(SUM(won) / COUNT(game_id) * 100, 1) AS win_pct,
        ROUND(AVG(goals), 2)         AS avg_goals,
        ROUND(AVG(shots), 2)         AS avg_shots,
        ROUND(AVG(hits), 2)          AS avg_hits,
        ROUND(AVG(faceoff_win_pct), 2) AS avg_faceoff_win_pct,
        ROUND(AVG(shot_pct), 2)      AS avg_shot_pct
    FROM fact_game_performance
    WHERE type = 'R'
    GROUP BY season, home_or_away
    ORDER BY season, home_or_away
""")
home_away.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('gold_home_away_summary')
print('gold_home_away_summary saved ✅')

In [ ]:
# ── KPI 4: Goal Scoring by Period ─────────────────────────────────────────────
goals_by_period = spark.sql("""
    SELECT
        season,
        period,
        period_type,
        COUNT(*) AS total_goals,
        secondary_type AS shot_type
    FROM fact_play_events
    WHERE event = 'Goal'
    GROUP BY season, period, period_type, secondary_type
    ORDER BY season, period
""")
goals_by_period.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('gold_goals_by_period')
print('gold_goals_by_period saved ✅')

In [ ]:
# ── KPI 5: Power Play Efficiency ─────────────────────────────────────────────
pp_efficiency = spark.sql("""
    SELECT
        f.season,
        t.abbreviation,
        t.team_name,
        SUM(f.powerPlayOpportunities) AS pp_opportunities,
        SUM(f.powerPlayGoals)         AS pp_goals,
        ROUND(SUM(f.powerPlayGoals) / NULLIF(SUM(f.powerPlayOpportunities), 0) * 100, 2)
            AS pp_efficiency_pct
    FROM fact_game_performance f
    JOIN dim_team t ON f.team_id = t.team_id
    WHERE f.type = 'R'
    GROUP BY f.season, t.abbreviation, t.team_name
    ORDER BY f.season, pp_efficiency_pct DESC
""")
pp_efficiency.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('gold_powerplay_efficiency')
print('gold_powerplay_efficiency saved ✅')

---
## ✅ Phase 2 Complete — Gold Layer

| Gold Table | Description |
|------------|-------------|
| dim_team | Team dimension |
| dim_player | Player dimension |
| dim_date | Date dimension |
| fact_game_performance | Game-level team stats |
| fact_player_stats | Per-game player stats |
| fact_play_events | Play-by-play key events |
| gold_team_standings | Season standings KPI |
| gold_player_season_stats | Player season summary |
| gold_home_away_summary | Home vs away trends |
| gold_goals_by_period | Goals per period analysis |
| gold_powerplay_efficiency | Power play % per team |

**Next → Connect to Power BI & build dashboard (see dashboard guide)**